# Finetuned Model Evaluation

Analysis of finetuned models tested across datasets.
Currently: Ministral-8B finetuned on ABSTRCT, tested on all 10 datasets.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# Style configuration
plt.rcParams.update({"figure.dpi": 150, "axes.titlesize": 14, "axes.labelsize": 12})
sns.set_theme(style="whitegrid", palette="colorblind")

FINETUNED_DIR = Path(".").resolve()
print(f"Working directory: {FINETUNED_DIR}")

In [ ]:
# All 10 GAIC datasets
ALL_DATASETS = [
    "ABSTRCT", "ACQUA", "AEC", "AFS", "ARGUMINSCI",
    "FINARG", "IAM", "PE", "SCIARK", "USELEC"
]

def load_finetuned_results(results_dir: Path) -> list[dict]:
    """Load all finetuned model result files."""
    results = []
    for subdir in results_dir.iterdir():
        if subdir.is_dir() and not subdir.name.startswith("."):
            for json_file in subdir.glob("*.json"):
                with open(json_file) as f:
                    data = json.load(f)
                    # Extract training dataset from folder name (e.g., ministral_8b_ABSTRCT)
                    parts = subdir.name.split("_")
                    train_dataset = parts[-1] if len(parts) > 2 else "unknown"
                    data["_train_dataset"] = train_dataset
                    data["_model_name"] = "_".join(parts[:-1]) if len(parts) > 2 else subdir.name
                    data["_file"] = json_file.name
                    results.append(data)
    return results

results = load_finetuned_results(FINETUNED_DIR)
print(f"Loaded {len(results)} result files")
for r in results:
    print(f"  - {r['_model_name']} trained on {r['_train_dataset']}")

In [ ]:
def extract_metrics(result: dict) -> list[dict]:
    """Extract per-dataset metrics from a result file."""
    rows = []
    train_dataset = result.get("_train_dataset", "unknown")
    model_name = result.get("_model_name", "unknown")
    
    for dataset, metrics in result.get("datasets", {}).items():
        reports = metrics.get("reports", {})
        for manipulation in ["original", "content_only", "shuffle"]:
            report = reports.get(manipulation, {})
            macro_f1 = report.get("macro avg", {}).get("f1-score", np.nan)
            accuracy = report.get("accuracy", np.nan)
            
            # Per-class metrics
            arg_f1 = report.get("Argument", {}).get("f1-score", np.nan)
            noarg_f1 = report.get("No-Argument", {}).get("f1-score", np.nan)
            arg_precision = report.get("Argument", {}).get("precision", np.nan)
            arg_recall = report.get("Argument", {}).get("recall", np.nan)
            
            rows.append({
                "train_dataset": train_dataset,
                "model": model_name,
                "test_dataset": dataset,
                "manipulation": manipulation,
                "macro_f1": macro_f1,
                "accuracy": accuracy,
                "arg_f1": arg_f1,
                "noarg_f1": noarg_f1,
                "arg_precision": arg_precision,
                "arg_recall": arg_recall,
                "is_in_domain": train_dataset == dataset,
            })
    return rows

# Build DataFrame
rows = []
for result in results:
    rows.extend(extract_metrics(result))

df = pd.DataFrame(rows)
print(f"Built DataFrame with {len(df)} rows")
df.head(10)

## Cross-Dataset Transfer Matrix

Rows = training dataset, Columns = test dataset.  
Diagonal = in-domain performance, off-diagonal = cross-dataset transfer.

In [ ]:
# Build transfer matrix: train_dataset x test_dataset
original_df = df[df["manipulation"] == "original"]

transfer_matrix = original_df.pivot_table(
    values="macro_f1",
    index="train_dataset",
    columns="test_dataset",
    aggfunc="mean"
)

# Reorder columns to match ALL_DATASETS order
cols_order = [d for d in ALL_DATASETS if d in transfer_matrix.columns]
transfer_matrix = transfer_matrix[cols_order]

print("Cross-Dataset Transfer Matrix (Macro-F1, original text)")
print("Rows = trained on, Columns = tested on")
transfer_matrix.round(3)

In [ ]:
# =============================================================================
# CROSS-DATASET TRANSFER HEATMAP (similar to Feger et al. style)
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 3 * len(transfer_matrix)))

# Plot heatmap
sns.heatmap(
    transfer_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0.3,
    vmax=0.9,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 12, "weight": "bold"},
    cbar_kws={"label": "Macro-F1", "shrink": 0.8}
)

# Highlight diagonal (in-domain) with thick border
for i, train_ds in enumerate(transfer_matrix.index):
    if train_ds in transfer_matrix.columns:
        j = list(transfer_matrix.columns).index(train_ds)
        ax.add_patch(Rectangle((j, i), 1, 1, fill=False,
                                edgecolor="blue", linewidth=3))

ax.set_title("Cross-Dataset Transfer Matrix\n(Trained on Row, Tested on Column)", 
             fontweight="bold", fontsize=14)
ax.set_xlabel("Test Dataset", fontsize=12)
ax.set_ylabel("Train Dataset", fontsize=12)

# Rotate x labels
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig(FINETUNED_DIR / "plot_transfer_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FINETUNED_DIR / 'plot_transfer_matrix.png'}")

## Performance Overview: In-Domain vs Out-of-Domain

In [ ]:
# Summary: in-domain vs out-of-domain performance
original_df = df[df["manipulation"] == "original"].copy()

summary = original_df.groupby(["train_dataset", "is_in_domain"])["macro_f1"].agg(["mean", "std", "count"])
summary.columns = ["Mean F1", "Std", "N Datasets"]
print("In-Domain vs Out-of-Domain Performance")
summary.round(3)

In [ ]:
# Bar chart: In-domain vs Out-of-domain
summary_reset = summary.reset_index()
summary_reset["Domain"] = summary_reset["is_in_domain"].map({True: "In-Domain", False: "Out-of-Domain"})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=summary_reset, x="train_dataset", y="Mean F1", hue="Domain",
            palette={"In-Domain": "#2ca02c", "Out-of-Domain": "#d62728"}, ax=ax)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3, fontsize=10)

ax.axhline(0.5, color="gray", linestyle="--", linewidth=1.5, label="Random baseline")
ax.set_title("In-Domain vs Out-of-Domain Performance", fontweight="bold")
ax.set_xlabel("Trained On")
ax.set_ylabel("Macro-F1")
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FINETUNED_DIR / "plot_domain_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Manipulation Sensitivity (Shortcut Learning Analysis)

In [ ]:
# Calculate deltas for each test dataset
def calculate_deltas(df: pd.DataFrame) -> pd.DataFrame:
    pivot = df.pivot_table(
        values="macro_f1",
        index=["train_dataset", "test_dataset", "is_in_domain"],
        columns="manipulation",
        aggfunc="mean"
    ).reset_index()
    
    if "original" in pivot.columns:
        if "content_only" in pivot.columns:
            pivot["delta_content_only"] = pivot["content_only"] - pivot["original"]
        if "shuffle" in pivot.columns:
            pivot["delta_shuffle"] = pivot["shuffle"] - pivot["original"]
    return pivot

deltas_df = calculate_deltas(df)
print("Deltas by Test Dataset")
print("Positive delta_content_only = WORSE when semantics removed = potential shortcut")
deltas_df.round(3)

In [ ]:
# Delta heatmap: test_dataset x manipulation delta
delta_pivot = deltas_df.pivot_table(
    values=["delta_content_only", "delta_shuffle"],
    index="test_dataset",
    aggfunc="mean"
)

# Reorder
delta_pivot = delta_pivot.reindex([d for d in ALL_DATASETS if d in delta_pivot.index])

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(
    delta_pivot,
    annot=True,
    fmt="+.2f",
    cmap="RdYlGn_r",  # Reversed: negative (expected) is green, positive (shortcut) is red
    vmin=-0.3,
    vmax=0.3,
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 11, "weight": "bold"},
    cbar_kws={"label": "Δ F1 (vs original)"}
)

ax.set_title("Manipulation Sensitivity by Test Dataset\n(Positive = BETTER when manipulated = shortcut learning)",
             fontweight="bold")
ax.set_xlabel("Manipulation")
ax.set_ylabel("Test Dataset")

plt.tight_layout()
plt.savefig(FINETUNED_DIR / "plot_delta_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Per-Dataset Performance Table

In [ ]:
# Detailed table: F1 by manipulation for each test dataset
manip_pivot = df.pivot_table(
    values="macro_f1",
    index=["train_dataset", "test_dataset"],
    columns="manipulation",
    aggfunc="mean"
)

# Reorder columns
col_order = ["original", "content_only", "shuffle"]
manip_pivot = manip_pivot[[c for c in col_order if c in manip_pivot.columns]]

# Add deltas
if "original" in manip_pivot.columns and "content_only" in manip_pivot.columns:
    manip_pivot["Δ_content"] = manip_pivot["content_only"] - manip_pivot["original"]
if "original" in manip_pivot.columns and "shuffle" in manip_pivot.columns:
    manip_pivot["Δ_shuffle"] = manip_pivot["shuffle"] - manip_pivot["original"]

print("F1 by Manipulation per Test Dataset")
styled = manip_pivot.style.background_gradient(
    subset=["original", "content_only", "shuffle"],
    cmap="RdYlGn", vmin=0.3, vmax=0.9
).background_gradient(
    subset=["Δ_content", "Δ_shuffle"],
    cmap="RdYlGn_r", vmin=-0.3, vmax=0.3
).format("{:.3f}")
display(styled)

## Class-Level Analysis (Argument vs No-Argument)

In [ ]:
# Check if model is biased towards one class
original_df = df[df["manipulation"] == "original"].copy()

class_pivot = original_df.pivot_table(
    values=["arg_f1", "noarg_f1", "arg_precision", "arg_recall"],
    index="test_dataset",
    aggfunc="mean"
)

# Reorder
class_pivot = class_pivot.reindex([d for d in ALL_DATASETS if d in class_pivot.index])

print("Per-Class Metrics (Original Text)")
print("High arg_recall + low arg_precision = predicting mostly Argument")
styled_class = class_pivot.style.background_gradient(
    cmap="RdYlGn", vmin=0.3, vmax=1.0
).format("{:.3f}")
display(styled_class)

In [ ]:
# Visualize class imbalance in predictions
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(class_pivot))
width = 0.35

bars1 = ax.bar(x - width/2, class_pivot["arg_f1"], width, label="Argument F1", color="#1f77b4")
bars2 = ax.bar(x + width/2, class_pivot["noarg_f1"], width, label="No-Argument F1", color="#ff7f0e")

ax.set_xlabel("Test Dataset")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Scores by Test Dataset\n(Large gap = class prediction bias)", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(class_pivot.index, rotation=45, ha="right")
ax.legend()
ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)

# Add value labels
ax.bar_label(bars1, fmt="%.2f", padding=3, fontsize=8)
ax.bar_label(bars2, fmt="%.2f", padding=3, fontsize=8)

plt.tight_layout()
plt.savefig(FINETUNED_DIR / "plot_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary Statistics

In [ ]:
# Overall summary
print("=" * 60)
print("FINETUNING EVALUATION SUMMARY")
print("=" * 60)

for train_ds in df["train_dataset"].unique():
    subset = df[(df["train_dataset"] == train_ds) & (df["manipulation"] == "original")]
    
    in_domain = subset[subset["is_in_domain"]]["macro_f1"].mean()
    out_domain = subset[~subset["is_in_domain"]]["macro_f1"].mean()
    
    print(f"\nTrained on: {train_ds}")
    print(f"  In-domain F1:      {in_domain:.3f}")
    print(f"  Out-of-domain F1:  {out_domain:.3f}")
    print(f"  Transfer gap:      {in_domain - out_domain:+.3f}")
    
    # Deltas
    in_domain_deltas = deltas_df[(deltas_df["train_dataset"] == train_ds) & (deltas_df["is_in_domain"])]
    if len(in_domain_deltas) > 0:
        dc = in_domain_deltas["delta_content_only"].mean()
        ds = in_domain_deltas["delta_shuffle"].mean()
        print(f"  In-domain Δ_content_only: {dc:+.3f}")
        print(f"  In-domain Δ_shuffle:      {ds:+.3f}")
        if dc > 0:
            print(f"  ⚠️  SHORTCUT WARNING: content_only IMPROVES performance!")

In [ ]:
# Export results
df.to_csv(FINETUNED_DIR / "results_finetuned.csv", index=False)
print(f"Exported: {FINETUNED_DIR / 'results_finetuned.csv'}")